# RAG-based Alignment — Analyse visuelle des résultats

Ce notebook charge les JSONs produits par les scripts d'éval (`scripts/eval_baseline.py`, `scripts/eval_rag.py`, `scripts/run_ablations.py`) et affiche :

1. **Tableau récapitulatif** baseline vs RAG (accuracy par catégorie + macro)
2. **Bar chart** par catégorie avec deltas
3. **Courbe d'ablation top_k**
4. **Heatmap** de désaccords baseline ↔ RAG (qui flippe vers où)
5. **Distribution des sources de chunks récupérés** par catégorie ETHICS
6. **Exemples qualitatifs** : prompt + chunks récupérés + prédictions

**Note** : les résultats par défaut (`results/baseline_cpu.json`, `results/rag_cpu.json`) viennent du smoke-test CPU (Qwen 0.5B, 20 ex par catégorie, 2 catégories). À remplacer par les JSONs GPU pour des chiffres comparables au rapport.

In [ ]:
import json
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RESULTS_DIR = Path('../results')

BASELINE_PATH = RESULTS_DIR / 'baseline_cpu.json'
RAG_PATH      = RESULTS_DIR / 'rag_cpu.json'
ABLATION_PATH = RESULTS_DIR / 'ablations_topk_cpu.json'

def load(p):
    p = Path(p)
    if not p.exists():
        print(f'!! missing {p}')
        return None
    return json.loads(p.read_text(encoding='utf-8'))

baseline = load(BASELINE_PATH)
rag      = load(RAG_PATH)
ablation = load(ABLATION_PATH)

print('Loaded:')
if baseline: print(f'  baseline — model={baseline["model"]}, mode={baseline["mode"]}')
if rag:      print(f'  rag      — backend={rag["retrieval_backend"]}, k={rag["top_k"]}, template={rag["template"]}')
if ablation: print(f'  ablation — {len(ablation["runs"])} runs')

## 1. Tableau récapitulatif

In [ ]:
def summary_table(baseline, rag):
    rows = []
    cats = list(baseline['details'].keys())
    for cat in cats:
        b = baseline['details'][cat]['accuracy']
        r = rag['details'][cat]['accuracy'] if rag else None
        n = baseline['details'][cat]['n']
        delta = (r - b) if r is not None else None
        rows.append({
            'category': cat,
            'n': n,
            'baseline_acc': round(b, 3),
            'rag_acc': round(r, 3) if r is not None else None,
            'delta_pp': round(delta * 100, 1) if delta is not None else None,
        })
    # macro-avg row
    b_macro = baseline['summary']['macro_avg']
    r_macro = rag['summary']['macro_avg'] if rag else None
    rows.append({
        'category': 'MACRO',
        'n': sum(r['n'] for r in rows),
        'baseline_acc': round(b_macro, 3),
        'rag_acc': round(r_macro, 3) if r_macro is not None else None,
        'delta_pp': round((r_macro - b_macro) * 100, 1) if r_macro is not None else None,
    })
    return pd.DataFrame(rows)

df_summary = summary_table(baseline, rag)
df_summary.style.format({'baseline_acc': '{:.1%}', 'rag_acc': '{:.1%}', 'delta_pp': '{:+.1f}'}).bar(
    subset=['delta_pp'], align='zero', color=['#d65f5f', '#5fba7d']
)

## 2. Bar chart par catégorie

In [ ]:
cats = list(baseline['details'].keys())
b_vals = [baseline['details'][c]['accuracy'] for c in cats]
r_vals = [rag['details'][c]['accuracy'] for c in cats] if rag else None

fig, ax = plt.subplots(figsize=(8, 4.5))
x = np.arange(len(cats))
w = 0.38
ax.bar(x - w/2, b_vals, w, label='Baseline (no RAG)', color='#8aa1c4')
if r_vals is not None:
    ax.bar(x + w/2, r_vals, w, label=f'RAG (k={rag["top_k"]})', color='#5fba7d')
ax.axhline(0.5, color='gray', linestyle=':', linewidth=1, label='chance (binary)')
ax.set_xticks(x)
ax.set_xticklabels(cats)
ax.set_ylabel('Accuracy')
ax.set_ylim(0, 1.0)
ax.set_title('ETHICS accuracy by category')
ax.legend(loc='lower right')
for i, v in enumerate(b_vals):
    ax.text(i - w/2, v + 0.01, f'{v:.0%}', ha='center', fontsize=9)
if r_vals is not None:
    for i, v in enumerate(r_vals):
        ax.text(i + w/2, v + 0.01, f'{v:.0%}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

## 3. Courbe d'ablation top_k

Comment l'accuracy évolue-t-elle quand on injecte plus de chunks dans le prompt ? Trop peu = pas assez de signal ; trop = bruit + dilution.

In [ ]:
if ablation is None:
    print('No ablation results yet. Run scripts/run_ablations.py --axes top_k')
else:
    runs = sorted(ablation['runs'], key=lambda r: r['top_k'])
    ks = [r['top_k'] for r in runs]
    macro = [r['summary']['macro_avg'] for r in runs]
    cats = list(runs[0]['summary']['per_category'].keys())

    fig, ax = plt.subplots(figsize=(8, 4.5))
    for cat in cats:
        vals = [r['summary']['per_category'][cat]['accuracy'] for r in runs]
        ax.plot(ks, vals, marker='o', label=cat, alpha=0.7)
    ax.plot(ks, macro, marker='s', linewidth=2.5, color='black', label='macro-avg')
    if baseline:
        ax.axhline(baseline['summary']['macro_avg'], color='#8aa1c4', linestyle='--',
                   label=f'baseline macro = {baseline["summary"]["macro_avg"]:.1%}')
    ax.set_xlabel('top_k (chunks retrieved per query)')
    ax.set_ylabel('Accuracy')
    ax.set_title('Accuracy vs. number of retrieved chunks')
    ax.set_xticks(ks)
    ax.legend(loc='best', fontsize=9)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    print('\nAblation runs:')
    df_abl = pd.DataFrame([
        {'top_k': r['top_k'], 'macro_acc': round(r['summary']['macro_avg'], 3),
         **{f'acc_{c}': round(r['summary']['per_category'][c]['accuracy'], 3) for c in cats}}
        for r in runs
    ])
    display(df_abl)

## 4. Heatmap des transitions baseline → RAG

Pour chaque catégorie, on regarde comment chaque exemple est classé par les deux systèmes. 4 cases possibles :
- **both correct** : les deux se sont accordés sur la bonne réponse
- **RAG fixes baseline** : la baseline se trompait, RAG corrige (gain)
- **RAG breaks baseline** : la baseline avait juste, RAG fait passer à faux (régression)
- **both wrong** : même erreur

Net gain = `fixes − breaks`.

In [ ]:
def transition_counts(baseline, rag, cat):
    b = baseline['details'][cat]
    r = rag['details'][cat]
    bp, rp, g = b['predictions'], r['predictions'], b['golds']
    both_ok = sum(int(bi == gi and ri == gi) for bi, ri, gi in zip(bp, rp, g))
    fixes   = sum(int(bi != gi and ri == gi) for bi, ri, gi in zip(bp, rp, g))
    breaks  = sum(int(bi == gi and ri != gi) for bi, ri, gi in zip(bp, rp, g))
    both_no = sum(int(bi != gi and ri != gi) for bi, ri, gi in zip(bp, rp, g))
    return both_ok, fixes, breaks, both_no

if rag is None:
    print('No RAG results.')
else:
    cats = list(baseline['details'].keys())
    mat = np.array([transition_counts(baseline, rag, c) for c in cats])
    labels = ['both ✓', 'RAG fixes', 'RAG breaks', 'both ✗']

    fig, ax = plt.subplots(figsize=(8, 1 + 0.5 * len(cats)))
    im = ax.imshow(mat, aspect='auto', cmap='RdYlGn',
                   vmin=-mat.max(), vmax=mat.max())
    # We want a colour gradient that says "more is good" for fixes/both_ok and
    # "more is bad" for breaks. Easier: just annotate the cells with counts.
    ax.set_xticks(range(4))
    ax.set_xticklabels(labels)
    ax.set_yticks(range(len(cats)))
    ax.set_yticklabels(cats)
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            ax.text(j, i, str(mat[i, j]), ha='center', va='center', fontsize=11,
                    color='black')
    ax.set_title('Per-category transitions: baseline → RAG')
    fig.colorbar(im, ax=ax, shrink=0.7, label='count')
    plt.tight_layout()
    plt.show()

    print('\nNet gain (fixes − breaks) per category:')
    for cat, (bo, fx, br, bn) in zip(cats, mat):
        print(f'  {cat:14s}  fixes={fx}  breaks={br}  net={fx - br:+d}')

## 5. Quelles sources sont retrouvées par catégorie ?

Si le retriever ramène des chunks `udhr.md` pour des questions ETHICS/justice et `virtue.md` pour ETHICS/virtue, c'est qu'il route correctement. Si tout pointe vers la même source, le corpus est mal exploité.

In [ ]:
if rag is None:
    print('No RAG results.')
else:
    cats = list(rag['details'].keys())
    sources_per_cat = {}
    for cat in cats:
        ids = rag['details'][cat]['retrieved_ids']
        flat = [cid.split('#')[0] for example_ids in ids for cid in example_ids]
        sources_per_cat[cat] = Counter(flat)

    all_sources = sorted({s for c in sources_per_cat.values() for s in c})
    mat = np.array([
        [sources_per_cat[cat].get(s, 0) for s in all_sources]
        for cat in cats
    ], dtype=float)
    # Normalize per category so columns are comparable across categories.
    row_sums = mat.sum(axis=1, keepdims=True)
    mat_norm = np.divide(mat, row_sums, out=np.zeros_like(mat), where=row_sums > 0)

    fig, ax = plt.subplots(figsize=(max(8, 1 + 0.7 * len(all_sources)), 1.5 + 0.6 * len(cats)))
    im = ax.imshow(mat_norm, aspect='auto', cmap='YlGnBu', vmin=0, vmax=mat_norm.max())
    ax.set_xticks(range(len(all_sources)))
    ax.set_xticklabels(all_sources, rotation=30, ha='right')
    ax.set_yticks(range(len(cats)))
    ax.set_yticklabels(cats)
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            n_raw = int(mat[i, j])
            ax.text(j, i, str(n_raw), ha='center', va='center', fontsize=9,
                    color='white' if mat_norm[i, j] > mat_norm.max() / 2 else 'black')
    ax.set_title('Retrieved sources by ETHICS category (raw counts, row-normalized colour)')
    fig.colorbar(im, ax=ax, shrink=0.7, label='share of retrievals (row-norm)')
    plt.tight_layout()
    plt.show()

## 6. Exemples qualitatifs : prompt + chunks récupérés + prédictions

On affiche quelques cas où RAG **corrige** la baseline et quelques cas où RAG **casse** une bonne réponse. Avec les chunks récupérés, on peut diagnostiquer pourquoi.

In [ ]:
from IPython.display import Markdown, display

# We need to rebuild the actual ETHICS prompts to show them. Easiest: import
# the same loader the eval used. Notebook lives in RAGbased/notebooks/, so add
# the parent directory to sys.path.
import sys
sys.path.insert(0, str(Path('..').resolve()))

from src.data import iter_all_ethics
from src.corpus import load_corpus

if rag is None:
    print('No RAG results.')
else:
    examples_by_cat = iter_all_ethics(
        list(rag['details'].keys()),
        baseline['details'][list(rag['details'].keys())[0]]['n'],
        seed=42,
    )
    chunks = load_corpus('../corpus')
    chunk_by_id = {c.id: c for c in chunks}

    def show_examples(kind: str, n: int = 3):
        """kind in {'fixes', 'breaks', 'both_wrong'}"""
        shown = 0
        for cat, ex_list in examples_by_cat.items():
            bp = baseline['details'][cat]['predictions']
            rp = rag['details'][cat]['predictions']
            g  = baseline['details'][cat]['golds']
            ret_ids = rag['details'][cat]['retrieved_ids']
            for i, ex in enumerate(ex_list):
                b_ok = bp[i] == g[i]
                r_ok = rp[i] == g[i]
                match = (
                    (kind == 'fixes' and not b_ok and r_ok) or
                    (kind == 'breaks' and b_ok and not r_ok) or
                    (kind == 'both_wrong' and not b_ok and not r_ok)
                )
                if not match:
                    continue
                md = [f'### [{cat}] example #{i} — gold={"Yes" if g[i] else "No"}']
                md.append(f'- **baseline** → {"Yes" if bp[i] else "No"} {"✓" if b_ok else "✗"}')
                md.append(f'- **rag**      → {"Yes" if rp[i] else "No"} {"✓" if r_ok else "✗"}')
                md.append('')
                md.append('**Prompt:**')
                md.append('```')
                md.append(ex.prompt)
                md.append('```')
                md.append('')
                md.append('**Retrieved chunks:**')
                for cid in ret_ids[i]:
                    c = chunk_by_id.get(cid)
                    if c is None:
                        md.append(f'- `{cid}` *(not found in corpus)*')
                    else:
                        body = c.text.replace('\n', ' ')[:220]
                        md.append(f'- **{cid}** — {body}...')
                display(Markdown('\n'.join(md)))
                shown += 1
                if shown >= n:
                    return
        if shown == 0:
            print(f'(no {kind} examples found)')

    print('\n=== CASES WHERE RAG FIXES BASELINE ===\n')
    show_examples('fixes', n=3)

    print('\n=== CASES WHERE RAG BREAKS A CORRECT BASELINE ===\n')
    show_examples('breaks', n=2)

## 7. Diagnostic des log-probabilités

Au lieu de juste regarder Yes/No prédit, on peut comparer `logP(Yes) − logP(No)` entre baseline et RAG. Si RAG augmente la marge dans la bonne direction (plus positive quand la gold = 1, plus négative quand la gold = 0), c'est un signe que le contexte aide même quand la prédiction finale ne change pas.

In [ ]:
# logprobs are only stored in the eval JSON via the EvalResult dataclass, but
# eval_baseline.py / eval_rag.py currently dump only predictions+golds. Re-run
# them after extending the JSON if you want this plot; here we show what we
# have so the cell is non-fatal.

def has_logprobs(eval_obj):
    return eval_obj and any(
        'yes_logprobs' in d for d in eval_obj['details'].values()
    )

if has_logprobs(baseline) and has_logprobs(rag):
    fig, axes = plt.subplots(1, len(baseline['details']), figsize=(5 * len(baseline['details']), 4), sharey=True)
    if len(baseline['details']) == 1:
        axes = [axes]
    for ax, cat in zip(axes, baseline['details']):
        b = baseline['details'][cat]
        r = rag['details'][cat]
        b_margin = np.array(b['yes_logprobs']) - np.array(b['no_logprobs'])
        r_margin = np.array(r['yes_logprobs']) - np.array(r['no_logprobs'])
        golds = np.array(b['golds'])
        for label, color in [(1, '#5fba7d'), (0, '#d65f5f')]:
            mask = golds == label
            ax.scatter(b_margin[mask], r_margin[mask], c=color, alpha=0.7,
                       label=f'gold = {"Yes" if label else "No"}')
        lim = max(abs(b_margin).max(), abs(r_margin).max()) * 1.05
        ax.plot([-lim, lim], [-lim, lim], 'k:', alpha=0.4)
        ax.axhline(0, color='gray', linewidth=0.5)
        ax.axvline(0, color='gray', linewidth=0.5)
        ax.set_xlabel('baseline: logP(Yes) − logP(No)')
        ax.set_ylabel('rag: logP(Yes) − logP(No)')
        ax.set_title(f'{cat}')
        ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()
else:
    print('Log-probabilities not stored in the eval JSON. '
          'Extend eval_baseline.py / eval_rag.py to dump `yes_logprobs` and `no_logprobs` to use this cell.')

## 8. Récap textuel

Une cellule terminale qui résume tout en une phrase. Utile à coller dans le rapport.

In [ ]:
if baseline and rag:
    b_macro = baseline['summary']['macro_avg']
    r_macro = rag['summary']['macro_avg']
    delta = (r_macro - b_macro) * 100
    cats = list(baseline['details'].keys())
    per_cat = []
    for c in cats:
        b = baseline['details'][c]['accuracy']
        r = rag['details'][c]['accuracy']
        per_cat.append(f'{c}: {b:.0%} → {r:.0%} ({(r-b)*100:+.1f}pp)')
    summary = (
        f'Model: {baseline["model"]}\n'
        f'Retriever: {rag["retrieval_backend"]} (k={rag["top_k"]}, template={rag["template"]})\n'
        f'Macro-accuracy: baseline {b_macro:.1%} → RAG {r_macro:.1%}  ({delta:+.1f}pp)\n'
        f'Per category:\n  ' + '\n  '.join(per_cat)
    )
    print(summary)
else:
    print('Missing results. Run scripts/eval_baseline.py and scripts/eval_rag.py.')